# Final experiments

Full-dataset experiments: 668 train / 118 validation / 232 test (93 round-1 gold + 139 round-2 shared). Runs the TF-IDF baseline with a learning curve, XLM-R and MuRIL across three random seeds, SHAP explanations, and the retrieval evaluation.

## Setup and GPU check

In [ ]:
!pip install -q shap sentence-transformers faiss-cpu
!nvidia-smi -L

## Checkpointing
`USE_DRIVE = True` saves each finished run to Google Drive so training can resume after a disconnect.

In [ ]:
USE_DRIVE = True
import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_DIR = "/content/drive/MyDrive/hinglish_absa_runs"
else:
    SAVE_DIR = "/content/runs"
os.makedirs(SAVE_DIR, exist_ok=True)
print("Saving to:", SAVE_DIR)

## Load dataset

In [ ]:
from google.colab import files
if not os.path.exists("final_dataset.csv"):
    files.upload()

## Core utilities: loading, scoring, baseline, learning curve

In [ ]:
import json, os, random
import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

CATS = ["Delivery", "Price_Charges", "Offers_Discounts", "Product_Quality", "Availability",
        "Wrong_Missing_Item", "Refund_Return", "Payment_Wallet", "Customer_Support",
        "App_Experience", "Packaging", "Overall"]
NO_OVERALL = [c for c in CATS if c != "Overall"]
LABELS = ["None", "Positive", "Negative", "Neutral"]


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)


def load_final(path="final_dataset.csv"):
    d = pd.read_csv(path)
    for c in CATS:
        d[c] = d[c].fillna("None")
    d["mask_overall"] = d["mask_overall"].astype(str).str.lower().eq("true")
    train, val, test = (d[d.split == s].reset_index(drop=True) for s in ["train", "val", "test"])
    return d, train, val, test


def usable(df, c):
    """Rows whose label for category c can be trained on (masks old-rule Overall labels)."""
    return df[~((c == "Overall") & df["mask_overall"])]


# ------------------------------------------------------------------ metrics
def _prf(pred, gold):
    tp = len(pred & gold)
    p = tp / len(pred) if pred else 0.0
    r = tp / len(gold) if gold else 0.0
    return p, r, (2 * p * r / (p + r) if p + r else 0.0)


def evaluate(gold_df, pred_df, cats=None):
    cats = cats or CATS
    g_det, p_det, g_full, p_full, per_cat = set(), set(), set(), set(), []
    for c in cats:
        g_c, p_c = set(), set()
        for i in range(len(gold_df)):
            rid, g, p = gold_df.iloc[i]["ID"], gold_df.iloc[i][c], pred_df.iloc[i][c]
            if g != "None":
                g_det.add((rid, c)); g_full.add((rid, c, g)); g_c.add((rid, g))
            if p != "None":
                p_det.add((rid, c)); p_full.add((rid, c, p)); p_c.add((rid, p))
        if g_c:
            per_cat.append({"Category": c, "F1": _prf(p_c, g_c)[2], "Gold mentions": len(g_c)})
    dp, dr, df = _prf(p_det, g_det)
    fp, fr, ff = _prf(p_full, g_full)
    return ({"Detection P": dp, "Detection R": dr, "Detection F1": df,
             "Aspect+Sentiment P": fp, "Aspect+Sentiment R": fr, "Aspect+Sentiment F1": ff,
             "Macro F1": float(np.mean([x["F1"] for x in per_cat])) if per_cat else 0.0},
            pd.DataFrame(per_cat))


def summarise(runs):
    """runs: list of metric dicts (one per seed) -> 'mean ± std' strings and raw means."""
    df = pd.DataFrame(runs)
    mean, std = df.mean(), df.std(ddof=0)
    return {k: f"{mean[k]:.3f} ± {std[k]:.3f}" for k in df.columns}, mean.to_dict()


# ------------------------------------------------------------------ baseline
def tfidf_baseline(train, test):
    word = TfidfVectorizer(ngram_range=(1, 2), lowercase=True)
    char = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, lowercase=True)
    Xtr = hstack([word.fit_transform(train.Review), char.fit_transform(train.Review)]).tocsr()
    Xte = hstack([word.transform(test.Review), char.transform(test.Review)]).tocsr()
    pred = test[["ID"]].copy()
    for c in CATS:
        keep = train.index.isin(usable(train, c).index)
        y = train[c].values[keep]
        if len(set(y)) < 2:
            pred[c] = y[0]
            continue
        clf = LogisticRegression(max_iter=3000, class_weight="balanced", random_state=0)
        clf.fit(Xtr[keep], y)
        pred[c] = clf.predict(Xte)
    return pred


def learning_curve(train_pool, test, fractions=(0.125, 0.25, 0.5, 0.75, 1.0), repeats=5):
    rows = []
    for f in fractions:
        for r in range(1 if f == 1.0 else repeats):
            sub = train_pool.sample(frac=f, random_state=r) if f < 1.0 else train_pool
            m, _ = evaluate(test, tfidf_baseline(sub.reset_index(drop=True), test))
            rows.append({"fraction": f, "n_train": len(sub), "F1": m["Aspect+Sentiment F1"]})
    return pd.DataFrame(rows).groupby(["fraction", "n_train"]).F1.agg(["mean", "std"]).reset_index().fillna(0)


## Load and split data

In [ ]:
data, train, val, test = load_final("final_dataset.csv")
print(f"Train {len(train)} | Validation {len(val)} | Test {len(test)}")
stats = pd.DataFrame({c: {s: int((data[c] == s).sum()) for s in ["Positive", "Negative", "Neutral"]} for c in CATS}).T
stats["Total"] = stats.sum(axis=1)
stats["In test"] = [int((test[c] != "None").sum()) for c in CATS]
print(stats.to_string())

## TF-IDF baseline and learning curve

In [ ]:
pool = pd.concat([train, val]).reset_index(drop=True)
pred_tfidf = predict_base = tfidf_baseline(pool, test)
base_all, base_pc = evaluate(test, pred_tfidf)
base_no, _ = evaluate(test, pred_tfidf, NO_OVERALL)
print("All 12:", {k: round(v, 3) for k, v in base_all.items()})
print("Without Overall:", {k: round(v, 3) for k, v in base_no.items()})
curve = learning_curve(pool, test)
print("\nLearning curve (Aspect+Sentiment F1):"); print(curve.round(3).to_string(index=False))
curve.to_csv(f"{SAVE_DIR}/learning_curve.csv", index=False)

## Transformer utilities
Class-weighted loss, to prevent the majority 'None' class from dominating training.

In [ ]:
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
L2I = {l: i for i, l in enumerate(LABELS)}
print("Using:", DEVICE)


def cat_text(c):
    return c.replace("_", " ").lower()


def to_pairs(df, none_ratio=3, seed=0):
    rows = [(r["Review"], cat_text(c), L2I[r[c]]) for _, r in df.iterrows() for c in CATS
            if not (c == "Overall" and r["mask_overall"])]
    pos = [x for x in rows if x[2] != 0]
    neg = [x for x in rows if x[2] == 0]
    rng = random.Random(seed)
    rows = pos + rng.sample(neg, min(len(neg), none_ratio * len(pos)))
    rng.shuffle(rows)
    return rows


def class_weights(rows):
    """Gentle balancing: weight = sqrt(total / (4 * count)). Stops the model from predicting 'None' for everything."""
    counts = np.bincount([r[2] for r in rows], minlength=len(LABELS)).astype(float)
    w = np.sqrt(counts.sum() / (len(LABELS) * np.maximum(counts, 1)))
    return torch.tensor(w / w.mean(), dtype=torch.float)


@torch.no_grad()
def predict_proba(model, tok, texts, cat, max_len=128, bs=64):
    model.eval()
    out, texts = [], [str(t) for t in texts]
    for i in range(0, len(texts), bs):
        chunk = texts[i:i + bs]
        enc = tok(chunk, [cat_text(cat)] * len(chunk), truncation=True, max_length=max_len,
                  padding=True, return_tensors="pt").to(DEVICE)
        out.append(torch.softmax(model(**enc).logits.float(), -1).cpu().numpy())
    return np.concatenate(out)


def predict_transformer(model, tok, df):
    pred = df[["ID"]].copy()
    for c in CATS:
        pred[c] = [LABELS[i] for i in predict_proba(model, tok, df.Review.tolist(), c).argmax(1)]
    return pred


def train_transformer(model_name, train, val, seed, epochs=10, lr=3e-5, bs=16, max_len=128):
    set_seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(LABELS)).to(DEVICE)
    rows = to_pairs(train, seed=seed)
    loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights(rows).to(DEVICE))

    def collate(batch):
        enc = tok([b[0] for b in batch], [b[1] for b in batch], truncation=True,
                  max_length=max_len, padding=True, return_tensors="pt")
        enc["labels"] = torch.tensor([b[2] for b in batch])
        return enc

    dl = DataLoader(rows, batch_size=bs, shuffle=True, collate_fn=collate,
                    generator=torch.Generator().manual_seed(seed))
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    steps = epochs * len(dl)
    sch = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
    best_f1, best_state, log = -1, None, []
    for ep in range(epochs):
        model.train(); total = 0.0
        for b in dl:
            b = {k: v.to(DEVICE) for k, v in b.items()}
            labels = b.pop("labels")
            loss = loss_fn(model(**b).logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sch.step(); opt.zero_grad()
            total += loss.item()
        m, _ = evaluate(val, predict_transformer(model, tok, val))
        log.append({"model": model_name, "seed": seed, "epoch": ep + 1,
                    "train_loss": round(total / len(dl), 4), "val_F1": round(m["Aspect+Sentiment F1"], 4)})
        print(log[-1])
        if m["Aspect+Sentiment F1"] > best_f1:
            best_f1 = m["Aspect+Sentiment F1"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    if best_f1 <= 0:
        print(f"WARNING: {model_name} (seed {seed}) never predicted a correct aspect on validation.")
    model.load_state_dict(best_state); model.to(DEVICE)
    return model, tok, log, best_f1


## Train XLM-R and MuRIL across three seeds
Each completed run is checkpointed; the cell skips runs already saved.

In [ ]:
MODELS = {"XLM-R": ("xlm-roberta-base", 3e-5), "MuRIL": ("google/muril-base-cased", 3e-5)}
SEEDS = [13, 42, 2026]
best = {"val_F1": -1}
for name, (hf_name, lr) in MODELS.items():
    for seed in SEEDS:
        tag = f"{name}_seed{seed}"
        if os.path.exists(f"{SAVE_DIR}/{tag}.json"):
            print("Already done:", tag); continue
        model, tok, log, val_f1 = train_transformer(hf_name, train, val, seed, lr=lr)
        pred = predict_transformer(model, tok, test)
        m_all, pc = evaluate(test, pred)
        m_no, _ = evaluate(test, pred, NO_OVERALL)
        json.dump({"model": name, "seed": seed, "val_F1": val_f1, "all": m_all, "no_overall": m_no,
                   "per_cat": pc.to_dict(orient="records"), "log": log}, open(f"{SAVE_DIR}/{tag}.json", "w"))
        pred.to_csv(f"{SAVE_DIR}/{tag}_test_pred.csv", index=False)
        print(tag, "test Aspect+Sentiment F1:", round(m_all["Aspect+Sentiment F1"], 3))
        if val_f1 > best["val_F1"]:
            if "model" in best: del best["model"]; torch.cuda.empty_cache()
            best = {"val_F1": val_f1, "name": name, "seed": seed, "hf": hf_name, "lr": lr, "model": model, "tok": tok}
        else:
            del model; torch.cuda.empty_cache()

## Aggregate results
Mean +/- standard deviation across seeds.

In [ ]:
import glob
runs = [json.load(open(f)) for f in sorted(glob.glob(f"{SAVE_DIR}/*_seed*.json"))]
fmt = lambda d: {k: f"{v:.3f}" for k, v in d.items()}
table_all = {"TF-IDF + LR": fmt(base_all)}
table_no = {"TF-IDF + LR": fmt(base_no)}
per_cat = {"TF-IDF + LR": base_pc.to_dict(orient="records")}
for name in MODELS:
    rs = [r for r in runs if r["model"] == name]
    if not rs: continue
    table_all[name] = summarise([r["all"] for r in rs])[0]
    table_no[name] = summarise([r["no_overall"] for r in rs])[0]
    pcs = pd.concat([pd.DataFrame(r["per_cat"]) for r in rs])
    per_cat[name] = pcs.groupby(["Category", "Gold mentions"]).F1.mean().reset_index().to_dict(orient="records")
cols = ["Detection F1", "Aspect+Sentiment F1", "Macro F1", "Aspect+Sentiment P", "Aspect+Sentiment R"]
print("=== All 12 categories ==="); print(pd.DataFrame(table_all).T[cols].to_string())
print("\n=== Without Overall ==="); print(pd.DataFrame(table_no).T[cols].to_string())
pd.DataFrame(table_all).T.to_csv(f"{SAVE_DIR}/table_all.csv"); pd.DataFrame(table_no).T.to_csv(f"{SAVE_DIR}/table_no_overall.csv")
best_run = max(runs, key=lambda r: r["val_F1"])
BEST_NAME = best_run["model"]
print("\nBest single run by validation:", BEST_NAME, "seed", best_run["seed"])
print("\nPer-category F1 (mean over seeds):")
print(pd.DataFrame({k: {x["Category"]: round(x["F1"], 3) for x in v} for k, v in per_cat.items()}).to_string())

## Load best model for explanation
Retrains the best run if it is no longer in memory after a disconnect.

In [ ]:
if best.get("name") == best_run["model"] and best.get("seed") == best_run["seed"] and "model" in best:
    model, tok = best["model"], best["tok"]
else:
    hf, lr = MODELS[best_run["model"]]
    model, tok, _, _ = train_transformer(hf, train, val, best_run["seed"], lr=lr)
pred_best = predict_transformer(model, tok, test)

# Export the exact validation-selected checkpoint for the live dashboard API.
from pathlib import Path
ARTIFACT_DIR = Path(SAVE_DIR) / "best_model"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(ARTIFACT_DIR), safe_serialization=True)
tok.save_pretrained(str(ARTIFACT_DIR))
manifest = {
    "schema_version": 1,
    "model_name": BEST_NAME,
    "base_model": MODELS[best_run["model"]][0],
    "seed": int(best_run["seed"]),
    "validation_aspect_sentiment_f1": float(best_run["val_F1"]),
    "categories": CATS,
    "labels": LABELS,
    "max_length": 128,
    "inference_batch_size": 32,
    "category_text_format": "category.replace('_', ' ').lower()",
    "selection": "highest validation Aspect+Sentiment F1 among completed model/seed runs",
}
(ARTIFACT_DIR / "manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
)
print("Dashboard API checkpoint exported to:", ARTIFACT_DIR)

## SHAP setup

In [ ]:
import shap


def _merge_words(pieces, values):
    """SHAP returns sub-word pieces; join them back into whole words and add up their scores."""
    words, new_word = [], True
    for p, v in zip(pieces, values):
        if not p.strip():
            new_word = True
            continue
        if new_word or p[:1].isspace() or not words:
            words.append([p.strip(), float(v)])
        else:
            words[-1][0] += p.strip()
            words[-1][1] += float(v)
        new_word = p[-1:].isspace()      # a piece ending in a space closes the word
    return [(w, round(v, 4)) for w, v in words]


def occlusion(model, tok, text, cat, idx):
    """Backup method: remove one word at a time and see how much the probability drops."""
    words = text.split()
    base = predict_proba(model, tok, [text], cat)[0, idx]
    variants = [" ".join(words[:i] + words[i + 1:]) for i in range(len(words))]
    probs = predict_proba(model, tok, variants, cat)[:, idx]
    return [(w, round(float(base - p), 4)) for w, p in zip(words, probs)]


def explain_reviews(model, tok, df, preds):
    """For each review and each aspect the model found, score every word."""
    out, method_used = {}, set()
    masker = shap.maskers.Text(tok)
    for i, r in df.iterrows():
        rid, text = r["ID"], r["Review"]
        found = [(c, preds.iloc[i][c]) for c in CATS if preds.iloc[i][c] != "None"]
        items = []
        for c, label in found:
            idx = L2I[label]
            try:
                f = lambda xs, c=c: predict_proba(model, tok, list(xs), c)
                sv = shap.Explainer(f, masker, output_names=LABELS)([text])
                words = _merge_words(list(sv.data[0]), sv.values[0][:, idx])
                method_used.add("SHAP")
            except Exception as e:
                words = occlusion(model, tok, text, c, idx)
                method_used.add("occlusion")
            items.append({"category": c, "label": label, "words": words})
        out[rid] = items
    print("Explanation method used:", method_used or "none")
    return out


def similar_reviews(data, past_ids, k=3):
    """Retrieval step of RAG: for every review, find the most similar labelled past reviews."""
    from sentence_transformers import SentenceTransformer
    import faiss
    enc = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
    E = enc.encode(data.Review.tolist(), normalize_embeddings=True, batch_size=64,
                   show_progress_bar=True).astype("float32")
    past_mask = data.ID.isin(past_ids).values
    past_idx = np.where(past_mask)[0]
    index = faiss.IndexFlatIP(E.shape[1])
    index.add(E[past_idx])
    sims, nn = index.search(E, k + 1)
    out = {}
    for i in range(len(data)):
        items = []
        for s, j in zip(sims[i], nn[i]):
            row = data.iloc[past_idx[j]]
            if row["ID"] == data.iloc[i]["ID"]:
                continue
            items.append({"id": row["ID"], "text": row["Review"], "similarity": round(float(s), 3),
                          "human": {c: row[c] for c in CATS if row[c] != "None"}})
        out[data.iloc[i]["ID"]] = items[:k]
    return out


## Explain test predictions

In [ ]:
N_EXPLAIN = 232
explanations = explain_reviews(model, tok, test.head(N_EXPLAIN).reset_index(drop=True), pred_best.head(N_EXPLAIN).reset_index(drop=True))
json.dump(explanations, open(f"{SAVE_DIR}/explanations.json", "w"), ensure_ascii=False)

## Retrieval and its evaluation

In [ ]:
from collections import Counter


def retrieval_eval(test, neighbours, k=3):
    """How useful are retrieved reviews? (a) share rate of a specific aspect, (b) F1 of copying neighbours' labels."""
    share = 0
    pred = test[["ID"]].copy()
    for c in CATS:
        pred[c] = "None"
    for i, r in test.iterrows():
        nb = neighbours.get(r["ID"], [])[:k]
        mine = {c for c in NO_OVERALL if r[c] != "None"}
        if nb and mine & {c for c in nb[0]["human"] if c != "Overall"}:
            share += 1
        for c in CATS:
            labs = [n["human"][c] for n in nb if c in n["human"]]
            if len(labs) >= 2:
                pred.loc[i, c] = Counter(labs).most_common(1)[0][0]
    m, _ = evaluate(test, pred)
    return {"top1_shares_aspect": share, "n_test": len(test),
            "knn_aspect_sentiment_F1": round(m["Aspect+Sentiment F1"], 3)}


def export_results(path, data, best_name, predictions, explanations, neighbours, table_all, table_no, per_cat):
    reviews = []
    for _, r in data.iterrows():
        rid = r["ID"]
        item = {"id": rid, "app": r["App"], "stars": None if pd.isna(r["stars"]) else int(r["stars"]),
                "text": r["Review"], "split": r["split"], "round": int(r["round"]),
                "human": {c: r[c] for c in CATS if r[c] != "None"}, "review_sentiment": r["Review_Sentiment"]}
        if rid in predictions: item["model"] = predictions[rid]
        if rid in explanations: item["explanation"] = explanations[rid]
        if rid in neighbours: item["similar"] = neighbours[rid]
        reviews.append(item)
    out = {"categories": CATS, "best_model": best_name,
           "metrics_all": table_all, "metrics_without_overall": table_no,
           "per_category": per_cat, "reviews": reviews}
    with open(path, "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=1, default=str)


past_ids = set(train.ID) | set(val.ID)
neighbours = similar_reviews(data, past_ids, k=3)
print(retrieval_eval(test, neighbours))

## Export final results

In [ ]:
all_pred = predict_transformer(model, tok, data)
predictions = {r.ID: {c: r[c] for c in CATS if r[c] != "None"} for _, r in all_pred.iterrows()}
export_results("results_v3.json", data, BEST_NAME, predictions, explanations, neighbours, table_all, table_no, per_cat)
import shutil
for f in ["table_all.csv", "table_no_overall.csv", "learning_curve.csv"]:
    shutil.copy(f"{SAVE_DIR}/{f}", f)
for f in ["results_v3.json", "table_all.csv", "table_no_overall.csv", "learning_curve.csv"]:
    files.download(f)